In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import *
import sys
sys.path.append("/Workspace/Users/sivana9908_gmail.com#ext#@sivana9908gmail.onmicrosoft.com/Uber-Eats-End-to-End-_Azure-Data-Engineering-Project")
from src.common.spark_utils import standardize_columns
from delta.tables import DeltaTable

In [0]:
#Reading data from adls 
df_menu = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://bronze@ubereaststorage.dfs.core.windows.net/sql/menu_items/")
)
display(df_menu)


In [0]:
# standardize columns
df_menu = standardize_columns(df_menu)

# standardize data types
df_menu = df_menu.withColumn("price", col("price").cast("int"))

#standardize actual data
df_menu=  df_menu.withColumn("is_available", upper(trim(col("is_available"))))
df_menu= df_menu.withColumn("category", upper(trim(col("category"))))
df_menu= df_menu.withColumn("item_name", upper(trim(col("item_name"))))

#handle null values
df_menu = df_menu.filter(col("restaurant_id").isNotNull())

#handle duplicates
df_menu = df_menu.dropDuplicates(["menu_item_id","restaurant_id"])

#apply business rules
df_menu = df_menu.filter(col("price") > 0)
df_menu =  df_menu.filter(col("is_available").isin("TRUE","FALSE"))

#validation
duplication_count=df_menu.groupBy("menu_item_id").count().filter(col("count")>1).count()
print("duplicate menu items",duplication_count)
price_notnull = df_menu.filter(col("price").isNull()).count()
print("price null",price_notnull)
isavialable_invalid = df_menu.filter(~col("is_available").isin("TRUE","FALSE")).count()










In [0]:
table_name= "ubereats_databricks1.silver.silver_menu_items"

if not spark.catalog.tableExists(table_name):

     df_menu .write.format("delta").mode("overwrite").saveAsTable("ubereats_databricks1.silver.silver_menu_items")   
      
else:

    target = DeltaTable.forName(spark, table_name)
    target.alias("t") \
        .merge(
            df_menu.alias("s"),
            "t.menu_item_id = s.menu_item_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
